# Import Mount Isa from OpenStreetMap

This demo keeps two local input options. The default uses OSMnx to pre-download and cache OSM responses in python/data/osm/osmnx_cache, then AequilibraE imports from that cache. After the first download, reuse or share that directory for a fast repeatable demo. The alternate option imports a prepared .osm.pbf file through pyrosm.

For the OSMnx cache option, no third-party extract service is used: OSMnx fetches the compact Mount Isa study area from OpenStreetMap services. The saved cache contains OSM HTTP response JSON, not a .osm.pbf file. Keep the cache directory and use the same study area, OSMnx version, and request settings when replaying it.

The live area is centered on the top-ranked Mount Isa town result with a 5 km radius, avoiding the much larger administrative boundaries returned by a full place-name search. OSM data is community maintained; retain attribution and review coverage before analysis.


In [ ]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import pandas as pd
from aequilibrae import Project

# A fresh folder lets the notebook be run more than once without replacing an existing project.
project_path = Path(gettempdir()) / f"aequilibrae_mount_isa_{uuid4().hex[:8]}"
print(f"AequilibraE project folder: {project_path}")


## Download and crop the Queensland PBF

This downloads the current Geofabrik Queensland extract, then uses Pyrosm to write a Mount Isa PBF crop. The extent matches the coordinates in the osmium example below; Pyrosm expects `[west, south, east, north]`, so the two latitude values are ordered from south to north. The source and crop are kept in `python/data/osm` (or `data/osm` when the notebook is launched from `python`).


In [ ]:
from urllib.request import urlretrieve

from pyrosm import OSM

osm_data_dir = Path.cwd() / "data" / "osm"
if not osm_data_dir.is_dir():
    osm_data_dir = Path.cwd() / "data" / "osm"
osm_data_dir.mkdir(parents=True, exist_ok=True)

geofabrik_url = "https://download.geofabrik.de/australia-oceania/australia/queensland-latest.osm.pbf"
source_pbf_path = osm_data_dir / "queensland-latest.osm.pbf"
pbf_path = osm_data_dir / "mount-isa.osm.pbf"

bbox = [139.45510, -20.79793, 139.57898, -20.60647]  # west, south, east, north
if not pbf_path.is_file():
    print(f"Downloading {geofabrik_url}")
    urlretrieve(geofabrik_url, source_pbf_path)
    print(f"Cropping Queensland PBF to {bbox}")
    OSM(source_pbf_path, bounding_box=bbox).to_pbf(
        output_path=str(pbf_path), repack=True
    )

print(f"Mount Isa PBF: {pbf_path}")

## Pre-download the OSM data

Set import_source to osmnx_cache to fetch the OSM response once into the persistent cache directory, or pbf to use a previously prepared local .osm.pbf file. The cache mode uses the same OSMnx query settings as the AequilibraE importer, so the import cell below reuses the cached response instead of making another Overpass request.


In [ ]:
import_source = "pbf"  # Choose "osmnx_cache" or "pbf".

if import_source not in {"osmnx_cache", "pbf"}:
    raise ValueError("import_source must be either 'osmnx_cache' or 'pbf'")

if import_source == "pbf":
    if not pbf_path.is_file():
        raise FileNotFoundError(f"Place a Mount Isa .osm.pbf extract at: {pbf_path}")
    print(f"Using local PBF: {pbf_path}")
else:
    import geopandas as gpd
    import osmnx as ox
    from shapely.geometry import Point

    # Keep OSMnx HTTP response files in a stable, shareable folder.
    ox.settings.use_cache = True
    ox.settings.cache_folder = osm_cache_dir

    place_name = "Mount Isa, Queensland, 4825, Australia"
    latitude, longitude = ox.geocode(place_name)
    buffer_m = 5_000
    centre = gpd.GeoSeries([Point(longitude, latitude)], crs="EPSG:4326")
    local_crs = centre.estimate_utm_crs()
    model_area = centre.to_crs(local_crs).buffer(buffer_m).to_crs("EPSG:4326").iloc[0]
    print(f"Study area: {buffer_m / 1_000:g} km radius in {local_crs}")

    # Warm the Overpass response cache before AequilibraE imports the same area.
    graph = ox.graph_from_polygon(
        model_area,
        network_type="all",
        simplify=False,
        retain_all=True,
        truncate_by_edge=False,
        custom_filter=None,
    )
    print(
        f"Downloaded graph: {len(graph.nodes):,} nodes, {len(graph.edges):,} directed edges"
    )
    del graph
    print(f"OSMnx response cache: {osm_cache_dir}")


## Import the cached data into AequilibraE
  With osmnx_cache, the AequilibraE importer repeats the same polygon query and OSMnx serves it from osmnx_cache. Once the cache has been prepared, this can run without reaching Nominatim or Overpass. With pbf, the importer reads the local file through pyrosm.


In [ ]:
project = Project()
project.new(project_path)

if import_source == "pbf":
    project.network.importer.osm(
        pbf_path=pbf_path,
        modes=("car", "transit", "bicycle", "walk"),
        simplify=True,
    )
else:
    project.network.importer.osm(
        model_area=model_area,
        modes=("car", "transit", "bicycle", "walk"),
        simplify=False,
    )


## Inspect the imported network

The project keeps its network in the AequilibraE project database. Inspect the imported link and node tables and check the modes represented in the network.

In [ ]:
links = project.network.links.data
nodes = project.network.nodes.data
print(f"Links: {len(links):,}\nNodes: {len(nodes):,}")
links.head()

In [ ]:
# zones_file = Path(
#     r"D:\OuterLoop\aequilibrae_training\python\data\Mount Isa SA1.parquet"
# )
zones_file = Path(
    "/mnt/c/Users/TylerPearn/Documents/aequilibrae_training/Mount Isa SA1.parquet"
)


# employment_file = Path(
#     r"D:\OuterLoop\aequilibrae_training\python\data\Mount Isa SA1 employment.csv"
# )
employment_file = Path(
    "/mnt/c/Users/TylerPearn/Documents/aequilibrae_training/Mount Isa SA1 employment.csv"
)


In [ ]:
zones = gpd.read_parquet(zones_file)[["SA1_CODE_2021", "geometry"]].explode()
zones = zones.assign(zone_id=range(1, len(zones) + 1))

zones["SA1_CODE_2021"] = zones["SA1_CODE_2021"].astype(int)
zones = zones.set_index("SA1_CODE_2021")
zones.head()

In [ ]:
employment = (
    pd.read_csv(employment_file, skiprows=9)  # Skip ABS header
    .tail(-1)  # Drop the first row which is blank for the two row header
    .head(-5)  # Drop the last 5 rows which contain data source info
    .loc[:, :"Total"]  # Drop columns after "Total", just whitespace
    .rename(
        columns={"1-digit level INDP Industry of Employment": "SA1_CODE_2021"}
    )  # Pandas reads this column name incorrectly due to the multi-row header
    .astype(int)
).set_index("SA1_CODE_2021")

column_names = {
    column: column.lower().replace(" ", "_").replace(",", "")
    for column in employment.columns
} | {"SA1_CODE_2021": "SA1_CODE_2021"}

employment = employment.rename(columns=column_names)

employment.head()

In [ ]:
zones = zones.join(employment, how="left")

In [ ]:
fields = project.network.zones.fields
existing_fields = fields.all_fields()

for proper_name, column in column_names.items():
    if column not in existing_fields:
        fields.add(column, description=f"ABS: {proper_name}", data_type="integer")
        print("Added field:", column)

fields.save()

In [ ]:
centroid_ids = project.network.zones.insert_from(zones.reset_index())

In [ ]:
project.network.zones.add_centroids()

In [ ]:
project.network.zones.connect_mode("b")

In [ ]:
from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap
import shapely
from shapely.geometry import box
import geopandas as gpd
from matplotlib import colormaps
import numpy as np

In [ ]:
from aequilibrae.paths.route_choice import RouteChoice

In [ ]:
# import numpy as np

In [ ]:
project.network.build_graphs(modes=["b"])
graph = project.network.graphs["b"]
graph.network["utility"] = 0.01 / 1000 * graph.network["distance"]

graph.prepare_graph(np.asarray(centroid_ids))
graph.set_graph("utility")

rc = RouteChoice(graph)
rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)

In [ ]:
rc.prepare([(15, 49)])

In [ ]:
rc.execute()

In [ ]:
routes = rc.get_results()

In [ ]:
routes

In [ ]:
links = project.network.links.data
nodes = project.network.nodes.data

In [ ]:
from functools import reduce

In [ ]:
route_links_ids = reduce(np.union1d, routes["route set"].abs())
route_links = links[links["link_id"].isin(route_links_ids)]

separate_routes_ids = list(routes["route set"].abs())
# [route_ids for route_ids in routes["route set"].abs()]
separate_routes_ids

In [ ]:
route_links_ids

In [ ]:
hover_links = links[
    [
        "link_id",
        "link_type",
        "direction",
        "geometry",
    ]
].rename(
    columns={
        "link_id": "Link ID",
        "link_type": "Link type",
        "direction": "Direction",
    }
)

link_layer = PathLayer.from_geopandas(
    hover_links,
    width_units="pixels",
    width_min_pixels=1,
    width_max_pixels=8,
    auto_highlight=True,
)
rc_layer = PathLayer.from_geopandas(
    route_links[["geometry"]],
    width_units="pixels",
    width_min_pixels=4,
    width_max_pixels=128,
    get_color=[0, 0, 255],
    pickable=False,
)

nodes_layer = ScatterplotLayer.from_geopandas(
    nodes[nodes["is_centroid"] != 1][
        [
            "node_id",
            "geometry",
        ]
    ],
    get_fill_color=[255, 0, 0],
    get_radius=10,
)
centroid_layer = ScatterplotLayer.from_geopandas(
    nodes[nodes["is_centroid"] == 1][
        [
            "node_id",
            "geometry",
        ]
    ],
    get_fill_color=[0, 255, 0],
    get_radius=20,
)

# 139.45510,-20.60647,139.57898,-20.79793
bbox = PolygonLayer.from_geopandas(
    gpd.GeoDataFrame(
        geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"
    ),
    get_fill_color=[100, 100, 100, 128],
)

traffic_map = Map(
    layers=[bbox, link_layer, rc_layer, nodes_layer, centroid_layer],
    basemap=None,
    height="1200px",
    show_side_panel=False,
    show_tooltip=True,
)
traffic_map

In [ ]:
# graph.prepare_graph(np.asarray(centroid_ids))
# graph.set_graph("distance")

# rc = RouteChoice(graph)
# rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)


def make_pretty_route_choice(
    source,
    dest,
    rc: RouteChoice,
    routes,
    project: Project,
    separate_routes: bool = True,
):
    routes = routes[not routes["mask"] != 1]
    routes = routes.sort_values(by="cost", ascending=False)  # sort descending
    links = project.network.links.data
    nodes = project.network.nodes.data

    route_links_ids = reduce(np.union1d, routes["route set"].abs())
    route_links = links[links["link_id"].isin(route_links_ids)]
    separate_routes_ids = list(routes["route set"].abs())
    separate_routes_links = [
        links[links["link_id"].isin(ids)] for ids in separate_routes_ids
    ]

    hover_links = links[
        [
            "link_id",
            "link_type",
            "direction",
            "geometry",
        ]
    ].rename(
        columns={
            "link_id": "Link ID",
            "link_type": "Link type",
            "direction": "Direction",
        }
    )

    link_layer = PathLayer.from_geopandas(
        hover_links,
        width_units="pixels",
        width_min_pixels=1,
        width_max_pixels=8,
        auto_highlight=True,
    )

    n_routes = len(separate_routes_links)
    normalized = np.linspace(0, 1, n_routes)
    turbo_colors = apply_continuous_cmap(normalized, colormaps["turbo"])
    # turbo_colors is an array of shape (n_routes, 4), values 0-255
    MIN_WIDTH, MAX_WIDTH = 4, 12  # pixels
    widths = np.linspace(MAX_WIDTH, MIN_WIDTH, n_routes)

    rc_layers: list[PathLayer] = []
    if separate_routes:
        for i, (single_route_links, color, width) in enumerate(
            zip(separate_routes_links, turbo_colors, widths)
        ):
            rc_layer = PathLayer.from_geopandas(
                single_route_links[["geometry"]],
                width_units="pixels",
                get_width=float(width),
                width_min_pixels=1,
                width_max_pixels=MAX_WIDTH,
                get_color=color.tolist(),
                pickable=False,
            )
            rc_layers.append(rc_layer)
    else:
        rc_layer = PathLayer.from_geopandas(
            route_links[["geometry"]],
            width_units="pixels",
            width_min_pixels=4,
            width_max_pixels=128,
            get_color=[0, 0, 255],
            pickable=False,
        )
        rc_layers.append(rc_layer)

    nodes_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] != 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[255, 0, 0],
        get_radius=10,
    )
    centroid_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] == 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[0, 255, 0],
        get_radius=20,
    )

    # 139.45510,-20.60647,139.57898,-20.79793
    bbox = PolygonLayer.from_geopandas(
        gpd.GeoDataFrame(
            geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"
        ),
        get_fill_color=[100, 100, 100, 128],
    )

    traffic_map = Map(
        layers=[bbox, link_layer, *rc_layers, nodes_layer, centroid_layer],
        basemap=None,
        height="1200px",
        show_side_panel=False,
        show_tooltip=True,
    )
    return traffic_map

In [ ]:
def inverse_binary_logit(prob, beta0, beta1):
    if prob == 1.0:
        return float("inf")
    elif prob == 0.0:
        return -float("inf")
    else:
        return (np.log(prob / (1.0 - prob)) - beta0) / beta1

In [ ]:
# graph.prepare_graph(np.asarray(centroid_ids))
# graph.set_graph("distance")

low_cutoff_rc = RouteChoice(graph)
my_cutoff = 0.0
low_cutoff_rc.set_choice_set_generation(
    algorithm="LP", max_routes=5, max_depth=5, cutoff_prob=my_cutoff
)

low_cutoff_rc.prepare([(14, 50)])
low_cutoff_rc.execute(perform_assignment=False)  # route choice sets only
routes = low_cutoff_rc.get_results()

routes


In [ ]:
inverse_logits = inverse_binary_logit((0.95 / 2) + 0.5, 0, 1)
inverse_logits

In [ ]:
make_pretty_route_choice(14, 50, low_cutoff_rc, routes, project, separate_routes=True)

In [ ]:
my_rc = RouteChoice(graph)
my_rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)

make_pretty_route_choice(my_rc, project)

In [ ]:
# Close the project database connection when you are finished.
# project.close()